In [ ]:
!pip install pypdf chromadb langchain langchain-community langchain-huggingface sentence-transformers

In [ ]:
from pypdf import PdfReader

pdf = PdfReader("bula_1782399219881.pdf")

texto = ""

for pagina in pdf.pages:
    texto += str(pagina.extract_text())

print("Tamanho do texto:", len(texto))

In [ ]:
print(texto[:1000])

In [ ]:
chunks = []

chunks.append("""
IDENTIFICAÇÃO DO MEDICAMENTO
Ibuprofeno 100mg
Suspensão oral
""")

chunks.append("""
PARA QUE SERVE
O ibuprofeno é usado para dor e febre.
""")

chunks.append("""
COMO FUNCIONA
Ação entre 15 e 30 minutos.
""")

chunks.append("""
QUANDO NÃO USAR
Alergia ao ibuprofeno ou menores de 6 meses.
""")

chunks.append("""
POSOLOGIA

Adultos: usar conforme orientação médica.
Crianças: seguir recomendação médica.
""")

chunks.append("""
EFEITOS COLATERAIS

Pode causar náusea, tontura e desconforto abdominal.
""")

chunks.append("""
INTERAÇÕES MEDICAMENTOSAS

Evitar uso concomitante com álcool.
""")

chunks.append("""
GRAVIDEZ E AMAMENTAÇÃO

Contraindicado no terceiro trimestre da gravidez.
""")

print("Chunks:", len(chunks))


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
print(len(chunks))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    persist_directory="chroma_db"
)

In [ ]:
query = "para que serve ibuprofeno?"

results = db.similarity_search(query, k=2)

for r in results:
    print("\n--- RESULTADO ---\n")
    print(r.page_content)

In [ ]:
def buscar_resposta(pergunta):

    resultados = db.similarity_search(pergunta, k=2)

    resposta = ""

    for doc in resultados:
        resposta += doc.page_content + "\n\n"

    return f"Segundo a bula:\n\n{resposta}"

In [ ]:
print(buscar_resposta("para que serve ibuprofeno"))

In [ ]:
while True:

    pergunta = input("Pergunta: ")

    if pergunta.lower() == "sair":
        break

    print(buscar_resposta(pergunta))